# Requirement
Please install `openai` along with generating an API Key for interacting with OpenAI platform
```
pip install openai
```

In [1]:
# Create a client to access the OpenAI API
from openai import OpenAI
client = OpenAI(api_key='')


In [ ]:
# Read base prompt from file
with open('./base_prompt.txt','r') as f:
    conent = f.read()

In [2]:
# Create a new assistant for assiting in annual report analysis
assistant = client.beta.assistants.create(
  name="Annual Report Analyst Assistant",
  instructions=conent,
  model="gpt-4o",
  tools=[{"type": "file_search"}],
)

In [3]:
# Upload the annual report file to openai for storing in vectorstore
message_file = client.files.create(
  file=open("./output/SFI_Baocaothuongnien_2022/SFI_Baocaothuongnien_2022.md", "rb"), purpose="assistants"
)

In [10]:
# Create a new thread with the annual report file attached
thread = client.beta.threads.create(
  messages=[
    {
      "role": "user",
      "content": "Read the attached annual report and list criteria for carbon disclosure.",
      "attachments": [
        { "file_id": message_file.id, "tools": [{"type": "file_search"}] }
      ],
    }
  ]
)

In [ ]:
# Execute the assistant to analyze the annual report
run = client.beta.threads.runs.create_and_poll(
    thread_id=thread.id, assistant_id=assistant.id
)

messages = list(client.beta.threads.messages.list(thread_id=thread.id, run_id=run.id))

message_content = messages[0].content[0].text
annotations = message_content.annotations
citations = []
for index, annotation in enumerate(annotations):
    message_content.value = message_content.value.replace(annotation.text, f"[{index}]")
    if file_citation := getattr(annotation, "file_citation", None):
        cited_file = client.files.retrieve(file_citation.file_id)
        citations.append(f"[{index}] {cited_file.filename}")

print(message_content.value)
print("\n".join(citations))